In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from scipy.stats import scoreatpercentile
from tqdm import tqdm
import time
import warnings

In [ ]:
# 1. 파일 이름 정의
FILE_NAME = 'smc_tidy.csv'

smc = pd.read_csv(FILE_NAME, dtype='str')
smc

In [ ]:
columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float

In [ ]:
dtype_int = {col: target_int for col in columns_to_int}
smc = smc.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
smc = smc.astype(dtype_float)

In [ ]:
# # ⭐ 사용할 컬럼 목록 정의 (20개)
ALL_COLUMNS_TO_USE = [
    'age', 'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_FEATURE = 'use_quan'
features = [c for c in ALL_COLUMNS_TO_USE if c != TARGET_FEATURE]

smc = smc[ALL_COLUMNS_TO_USE]

print("--- 데이터 로드 및 전처리 완료 ---\n")


In [ ]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 0. 설정
# -------------------------------------------------------------
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'
GROUP_COLUMN = 'procedure_cd_fin' # 그룹핑 기준 변수

smc = smc.copy()
# -------------------------------------------------------------
# 1. 수술별(procedure_cd_fin) IQR 및 임계값 계산
# -------------------------------------------------------------
print("수술별 그룹핑 및 IQR 계산 중...")

# (1) 각 수술 코드별로 Q1, Q3 계산하여 원본 데이터프레임에 임시 컬럼으로 붙임 (.transform 사용)
# transform은 그룹별로 계산한 값을 원래 행의 위치에 맵핑해줍니다.
smc['group_Q1'] = smc.groupby(GROUP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.quantile(0.25))
smc['group_Q3'] = smc.groupby(GROUP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.quantile(0.75))

# (2) 수술별 IQR 및 Upper Bound 계산
smc['group_IQR'] = smc['group_Q3'] - smc['group_Q1']
smc['group_upper_bound'] = smc['group_Q3'] + 1.5 * smc['group_IQR']

# (확인용) 수술별 임계값이 잘 들어갔는지 상위 5개만 출력
print(smc[[GROUP_COLUMN, TARGET_COLUMN, 'group_upper_bound', MSBOS_COLUMN]].head())

# -------------------------------------------------------------
# 2. 이상치 필터링 조건 생성 (Logic은 유지)
# -------------------------------------------------------------

# 조건 A: 해당 수술의 IQR 상한을 초과하는가?
# (전체 기준이 아니라, 내 수술 그룹의 상한선과 비교)
condition_A = (smc[TARGET_COLUMN] > smc['group_upper_bound'])

# 조건 B: MSBOS 값의 2배를 초과하는가? (MSBOS는 이미 수술별 값이므로 그대로 유지)
condition_B = (smc[TARGET_COLUMN] > 2 * smc[MSBOS_COLUMN])

# 최종 이상치 조건: A와 B를 동시에 만족할 때만 제거
outlier_condition = np.logical_and(condition_A, condition_B)

# -------------------------------------------------------------
# 3. 이상치 제거 및 정리
# -------------------------------------------------------------

# 이상치 개수 확인
num_outliers = outlier_condition.sum()

# 이상치 제거 (필터링)
smc_filtered = smc[~outlier_condition].copy()

# 계산을 위해 만들었던 임시 컬럼 삭제 (깔끔하게 정리)
smc_filtered.drop(columns=['group_Q1', 'group_Q3', 'group_IQR', 'group_upper_bound'], inplace=True, errors='ignore')

# 결과 출력
print("-" * 50)
print(f"제거 전략: [{GROUP_COLUMN}] 그룹별 IQR 적용")
print(f"제거된 이상치 데이터 개수: {num_outliers} 개")
print(f"원래 데이터셋 크기: {len(smc)} 행")
print(f"이상치 제거 후 데이터셋 크기: {len(smc_filtered)} 행")
print(f"데이터 감소율: {(num_outliers / len(smc) * 100):.2f}%")
print("-" * 50)


In [ ]:
threshold = 50
value_counts = smc_filtered['procedure_cd_fin'].value_counts()
frequent_codes = value_counts[value_counts >= threshold].index
smc_filtered = smc_filtered[smc_filtered['procedure_cd_fin'].isin(frequent_codes)]

In [ ]:
# scikit-learn의 train_test_split 함수를 사용하며, 층화를 위해 'stratify' 옵션을 사용합니다.
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------
# 1. 데이터 준비 (surgery9가 pandas DataFrame으로 로드되어 있다고 가정)
# -------------------------------------------------------------
# TARGET_COLUMN과 DROP_COLUMNS 설정은 이전 코드와 동일하게 유지
RANDOM_STATE = 2025


# 특성(X)과 종속 변수(y) 분리
X_smc = smc_filtered[features]
y_smc = smc_filtered[TARGET_FEATURE]

# 층화 기준 변수 설정
# One-Hot Encoding 등 전처리는 분할 후에 적용하는 것이 일반적입니다.
STRATA_VAR = smc_filtered['procedure_cd_fin']

# -------------------------------------------------------------
# 2. 층화 무작위 분할 수행 (70% Train, 30% Test)
# -------------------------------------------------------------

# stratify=STRATA_VAR 옵션을 사용하여 procedure_cd_fin의 비율을 Train/Test에 동일하게 유지합니다.
X_train_smc, X_test_smc, y_train_smc, y_test_smc = train_test_split(
    X_smc,
    y_smc,
    test_size=0.3, # 30%를 테스트 셋으로 사용
    random_state=RANDOM_STATE, # 무작위성 고정
    stratify=STRATA_VAR # procedure_cd_fin을 층화 기준으로 사용
)

# -------------------------------------------------------------
# 3. 결과 확인
# -------------------------------------------------------------
print("✅ Python의 train_test_split을 사용하여 층화 무작위 분할이 완료되었습니다.")
print(f"🔥 Train Data (특성) 크기: {X_train_smc.shape}")
print(f"🧪 Test Data (특성) 크기: {X_test_smc.shape}")

# (선택 사항) 층화 확인: procedure_cd_fin의 비율 비교
print("\n--- procedure_cd_fin 비율 비교 (Top 3) ---")
train_ratios = X_train_smc['procedure_cd_fin'].value_counts(normalize=True).head(3)
test_ratios = X_test_smc['procedure_cd_fin'].value_counts(normalize=True).head(3)
ratio_comparison = pd.DataFrame({'Train Ratio': train_ratios, 'Test Ratio': test_ratios})
print(ratio_comparison)

In [ ]:
# 2. 데이터 전처리 (제한된 컬럼에 맞춰 수정)
# ----------------------------------------------------------------------
def preprocess_data(df):
   
    # ⭐ 이진 (0/1) 플래그 컬럼 처리 ⭐
    binary_flag_cols = [
        'mi', 'pvd', 'mld', 'diabwc', 'rend', 'canc',
        'heparins', 'coumarin_deriatives'
    ]
   
    for col in binary_flag_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype(int)

    # ⭐ 범주형 변수  ⭐
    cat_features = ['surgeon_dept_cd', 'anesthesia_cd', 'procedure_cd_fin']
    for col in cat_features:
        if col in df.columns:
            df[col] = df[col].astype(str)

    target_procedures = ['5059'] # 평균 사용량 10이상인 수술 코드
    if 'procedure_cd_fin' in df.columns:
        for code in target_procedures:
            df[f'procedure_cd_fin_{code}'] = (df['procedure_cd_fin'] == code).astype(int)
        df = df.drop(columns = ['procedure_cd_fin'])
    df = pd.get_dummies(df, columns = ['surgeon_dept_cd', 'anesthesia_cd'])

    # ⭐ 수치형/임상 수치 컬럼 NaN 처리 ⭐
    numeric_features_for_imputation = ['age', 'Hb', 'PT_inr', 'Plt', 'aPTT']
   
    for col in numeric_features_for_imputation:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].mean())
       
    return df

X_train_smc = preprocess_data(X_train_smc)
X_test_smc = preprocess_data(X_test_smc)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


X_train_smc, X_test_smc = X_train_smc.align(X_test_smc, join='outer', axis = 1, fill_value = 0)

numeric_cols = ['age', 'Hb', 'PT_inr', 'Plt', 'aPTT']
cols_to_scale = [c for c in numeric_cols if c in X_train_smc.columns]

if cols_to_scale:
    scaler = StandardScaler()
    
    X_train_smc[cols_to_scale] = scaler.fit_transform(X_train_smc[cols_to_scale])
    X_test_smc[cols_to_scale] = scaler.transform(X_test_smc[cols_to_scale])



In [ ]:
X_train_smc = X_train_smc.astype(np.float32)
X_test_smc = X_test_smc.astype(np.float32)

In [ ]:
# diabwc 제거, 국소 마취 제거(5)
X_train_smc = X_train_smc[['Hb', 'PT_inr', 'Plt', 'aPTT', 'anesthesia_cd_1', 
       'anesthesia_cd_9', 'canc', 'heparins', 'msbos', 'pvd',
       'surgeon_dept_cd_ENT', 'surgeon_dept_cd_GS', 'surgeon_dept_cd_NS',
       'surgeon_dept_cd_OBGY', 'surgeon_dept_cd_OS', 'surgeon_dept_cd_PS',
       'surgeon_dept_cd_TS', 'surgeon_dept_cd_URO']]
X_test_smc = X_test_smc[['Hb', 'PT_inr', 'Plt', 'aPTT', 'anesthesia_cd_1',
       'anesthesia_cd_9', 'canc', 'heparins', 'msbos', 'pvd',
       'surgeon_dept_cd_ENT', 'surgeon_dept_cd_GS', 'surgeon_dept_cd_NS',
       'surgeon_dept_cd_OBGY', 'surgeon_dept_cd_OS', 'surgeon_dept_cd_PS',
       'surgeon_dept_cd_TS', 'surgeon_dept_cd_URO']]

n_features = len(list(X_train_smc.columns))

In [ ]:
def adjusted_r2(y_true, y_pred, n_features):
    """특성 개수를 고려한 Adjusted R-squared를 계산합니다."""
    r2 = r2_score(y_true, y_pred)
    n_samples = len(y_true)
   
    if n_samples > n_features + 1:
        adj_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
    else:
        adj_r2 = np.nan
       
    return adj_r2

def calculate_bootstrap_ci(model, X_test, y_test, n_bootstrap, confidence_level, n_features):
    random_seed = 2025
    np.random.seed(random_seed)
   
    lower_p = (100 - confidence_level) / 2
    upper_p = 100 - lower_p
   
    mae_scores, rmse_scores, r2_scores, adj_r2_scores = [], [], [], []
    n_test = len(X_test)
   
    print(f"      [부트스트랩] {n_bootstrap}회 실행 중...")
   
    for _ in tqdm(range(n_bootstrap)):
        sample_indices = np.random.choice(n_test, size=n_test, replace=True)
        X_sample = X_test.iloc[sample_indices]
        y_sample = y_test.iloc[sample_indices]
       
        y_pred = model.predict(X_sample)
        y_pred = np.maximum(y_pred, 0)
       
        mae_scores.append(mean_absolute_error(y_sample, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_sample, y_pred)))
        r2_scores.append(r2_score(y_sample, y_pred))
        adj_r2_scores.append(adjusted_r2(y_sample, y_pred, n_features))
       
    results = {
        'Model': [model.__class__.__name__],
        'Mean_RMSE': [np.mean(rmse_scores)], 'RMSE_CI_Lower': [scoreatpercentile(rmse_scores, lower_p)], 'RMSE_CI_Upper': [scoreatpercentile(rmse_scores, upper_p)],
        'Mean_MAE': [np.mean(mae_scores)], 'MAE_CI_Lower': [scoreatpercentile(mae_scores, lower_p)], 'MAE_CI_Upper': [scoreatpercentile(mae_scores, upper_p)],
        'Mean_R_Squared': [np.mean(r2_scores)], 'R2_CI_Lower': [scoreatpercentile(r2_scores, lower_p)], 'R2_CI_Upper': [scoreatpercentile(r2_scores, upper_p)],
        'Mean_Adj_R2': [np.nanmean(adj_r2_scores)],
        'Adj_R2_CI_Lower': [scoreatpercentile(np.array(adj_r2_scores)[~np.isnan(adj_r2_scores)], lower_p)],
        'Adj_R2_CI_Upper': [scoreatpercentile(np.array(adj_r2_scores)[~np.isnan(adj_r2_scores)], upper_p)],
    }
   
    return pd.DataFrame(results).round(4)

In [ ]:
# ----------------------------------------------------------------------
# 4. 모델 정의 및 확장된 튜닝 설정
# ----------------------------------------------------------------------
cv_strategy = RepeatedKFold(n_splits=10, n_repeats=10, random_state=2025)
scoring_metric = 'neg_root_mean_squared_error'

models = {
    # 1. 다중 선형 회귀 (튜닝 불필요)
    'Linear Regression': (LinearRegression(), {}),
   
    # 2. 랜덤 포레스트 (RF) - 확장된 튜닝 범위
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), {
        'n_estimators': [100, 300, 500], 
        'max_depth': [ 10, 20, 30],
        'min_samples_leaf': [1, 2, 4],
        'min_samples_split': [2, 5, 10]
    }),
   
    # 3. XGBoost - 확장된 튜닝 범위
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), {
        'n_estimators': [100, 500, 1000], 'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7, 9], 'subsample': [0.7, 1.0], 'colsample_bytree': [0.7, 1.0]
    }),
   
    # 4. SVR - 확장된 튜닝 범위
    'SVR': (SVR(kernel='rbf'), {
        'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto', 0.1]
    }),
   
    # 5. ANN (MLP) - 확장된 튜닝 범위
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), {
        'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)],
        'activation': ['relu', 'tanh'],
        'alpha': [0.0001, 0.001, 0.01], 'learning_rate_init': [0.001, 0.01]
    })
}


In [ ]:
#### SMC 모델
best_params_list = []
final_performance = pd.DataFrame()
predictions_df = pd.DataFrame({'Actual_Value': y_test_smc})

# ----------------------------------------------------------------------
# 5. 모델 학습, 예측 및 평가 실행 (훈련 성능 및 실행 시간 측정)
# ----------------------------------------------------------------------
print("--- 모델 학습 및 확장 튜닝 시작 ---\n")

for name, (model, param_grid) in models.items():
    print(f">>> {name} 학습 중...")
   
    start_time = time.time() # ⭐ 시간 측정 시작
   
    # 튜닝 및 학습
    if name == 'Linear Regression':
        trained_model = model.fit(X_train_smc, y_train_smc)
        best_params = 'Default'
    else:
        # ⭐ 튜닝 조합 수 계산 및 진행 정보 출력 ⭐
        param_counts = [len(v) for v in param_grid.values()]
        total_combinations = np.prod(param_counts)
       
        # --- CV 정보 고정 및 계산 (10-Fold, 10-Repeats 가정) ---
        n_splits_per_repeat = 10
        n_repeats = 10
        n_splits_total = n_splits_per_repeat * n_repeats # 100회
       
        # 총 Fit 횟수 계산 (총 조합 수 * 총 폴드 수)
        total_fits = total_combinations * n_splits_total
       
        print(f"\n      [CV 전략] {n_splits_per_repeat} K-Fold를 {n_repeats}회 반복 (총 {n_splits_total} 폴드)")
        print(f"      [Grid Search] 총 {total_combinations}개 조합 테스트 시작")
        print(f"      (전체 Fit 횟수: {total_fits}회)")
        
        grid_search = GridSearchCV(model, param_grid, cv=cv_strategy,
                                   scoring=scoring_metric, n_jobs=-1,verbose= 3)
        grid_search.fit(X_train_smc, y_train_smc)
        
        trained_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        
        best_params_list.append({
            'Model':name,
            'Best_Parameters': best_params
        })
        print(f"   Best Params: {grid_search.best_params_}")

    best_params_list.append({
        'model': name,
        'best_parameters': best_params
    })

    # ⭐ 훈련 데이터셋 성능 계산 및 출력 ⭐
    y_pred_train = trained_model.predict(X_train_smc)
    y_pred_train = np.maximum(y_pred_train, 0)
   
    train_rmse = np.sqrt(mean_squared_error(y_train_smc, y_pred_train))
    train_mae = mean_absolute_error(y_train_smc, y_pred_train)
    train_r2 = r2_score(y_train_smc, y_pred_train)
    train_adj_r2 = adjusted_r2(y_train_smc, y_pred_train, n_features)

    print(f"   [Train Set Performance]")
    print(f"     RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}, Adj_R2: {train_adj_r2:.4f}")
   
    # 테스트 데이터셋 부트스트랩 평가
    bootstrap_metrics_df = calculate_bootstrap_ci(trained_model, X_test_smc, y_test_smc,
                                                  n_bootstrap=100, confidence_level=95,
                                                  n_features = n_features)
   
    end_time = time.time() # ⭐ 시간 측정 종료
    execution_time = end_time - start_time
   
    # 최종 결과 DataFrame 업데이트
    bootstrap_metrics_df['Model'] = name
    bootstrap_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
   
    final_performance = pd.concat([final_performance, bootstrap_metrics_df], ignore_index=True)
   
    # 예측 결과 저장을 위한 예측 수행
    y_pred_full = trained_model.predict(X_test_smc)
    predictions_df[f'{name}_Prediction'] = y_pred_full
   
print("\n--- 모델 학습 및 부트스트랩 평가 완료 ---\n")


In [ ]:
pd.DataFrame(best_params_list).to_csv('smc_best_parmas.csv', index=False, encoding = 'utf-8-sig')
final_performance.to_csv('smc_best_performance.csv', index=False, encoding = 'utf-8-sig')

In [ ]:
# 1. 파일 이름 정의
FILE_NAME = 'amc_tidy.csv'

amc = pd.read_csv(FILE_NAME, encoding= 'utf-8', dtype='str')
amc

In [ ]:
amc.columns
amc = amc.rename(columns = {'anesthesia_cd2':'anesthesia_cd',
                          'use_quan_imputed':'use_quan'})

In [ ]:
columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float

In [ ]:
dtype_int = {col: target_int for col in columns_to_int}
amc = amc.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
amc = amc.astype(dtype_float)

In [ ]:
# ⭐ 사용할 컬럼 목록 정의 (20개)
ALL_COLUMNS_TO_USE = [
    'age', 'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_FEATURE = 'use_quan'

amc = amc[ALL_COLUMNS_TO_USE]
features = [c for c in ALL_COLUMNS_TO_USE if c != TARGET_FEATURE]

print("--- 데이터 로드 및 전처리 완료 ---\n")


In [ ]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 0. 설정
# -------------------------------------------------------------
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'
GROUP_COLUMN = 'procedure_cd_fin' # 그룹핑 기준 변수

amc = amc.copy()
# -------------------------------------------------------------
# 1. 수술별(procedure_cd_fin) IQR 및 임계값 계산
# -------------------------------------------------------------
print("수술별 그룹핑 및 IQR 계산 중...")

# (1) 각 수술 코드별로 Q1, Q3 계산하여 원본 데이터프레임에 임시 컬럼으로 붙임 (.transform 사용)
# transform은 그룹별로 계산한 값을 원래 행의 위치에 맵핑해줍니다.
amc['group_Q1'] = amc.groupby(GROUP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.quantile(0.25))
amc['group_Q3'] = amc.groupby(GROUP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.quantile(0.75))

# (2) 수술별 IQR 및 Upper Bound 계산
amc['group_IQR'] = amc['group_Q3'] - amc['group_Q1']
amc['group_upper_bound'] = amc['group_Q3'] + 1.5 * amc['group_IQR']

# (확인용) 수술별 임계값이 잘 들어갔는지 상위 5개만 출력
print(amc[[GROUP_COLUMN, TARGET_COLUMN, 'group_upper_bound', MSBOS_COLUMN]].head())

# -------------------------------------------------------------
# 2. 이상치 필터링 조건 생성 (Logic은 유지)
# -------------------------------------------------------------

# 조건 A: 해당 수술의 IQR 상한을 초과하는가?
# (전체 기준이 아니라, 내 수술 그룹의 상한선과 비교)
condition_A = (amc[TARGET_COLUMN] > amc['group_upper_bound'])

# 조건 B: MSBOS 값의 2배를 초과하는가? (MSBOS는 이미 수술별 값이므로 그대로 유지)
condition_B = (amc[TARGET_COLUMN] > 2 * amc[MSBOS_COLUMN])

# 최종 이상치 조건: A와 B를 동시에 만족할 때만 제거
outlier_condition = np.logical_and(condition_A, condition_B)

# -------------------------------------------------------------
# 3. 이상치 제거 및 정리
# -------------------------------------------------------------

# 이상치 개수 확인
num_outliers = outlier_condition.sum()

# 이상치 제거 (필터링)
amc_filtered = amc[~outlier_condition].copy()

# 계산을 위해 만들었던 임시 컬럼 삭제 (깔끔하게 정리)
amc_filtered.drop(columns=['group_Q1', 'group_Q3', 'group_IQR', 'group_upper_bound'], inplace=True, errors='ignore')

# 결과 출력
print("-" * 50)
print(f"제거 전략: [{GROUP_COLUMN}] 그룹별 IQR 적용")
print(f"제거된 이상치 데이터 개수: {num_outliers} 개")
print(f"원래 데이터셋 크기: {len(amc)} 행")
print(f"이상치 제거 후 데이터셋 크기: {len(amc_filtered)} 행")
print(f"데이터 감소율: {(num_outliers / len(amc) * 100):.2f}%")
print("-" * 50)


In [ ]:
threshold = 50
value_counts = amc_filtered['procedure_cd_fin'].value_counts()
frequent_codes = value_counts[value_counts >= threshold].index
amc_filtered = amc_filtered[amc_filtered['procedure_cd_fin'].isin(frequent_codes)]

In [ ]:
# scikit-learn의 train_test_split 함수를 사용하며, 층화를 위해 'stratify' 옵션을 사용합니다.
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------
# 1. 데이터 준비 (surgery9가 pandas DataFrame으로 로드되어 있다고 가정)
# -------------------------------------------------------------
# TARGET_COLUMN과 DROP_COLUMNS 설정은 이전 코드와 동일하게 유지
RANDOM_STATE = 2025


# 특성(X)과 종속 변수(y) 분리
X_amc = amc_filtered[features]
y_amc = amc_filtered[TARGET_FEATURE]

# 층화 기준 변수 설정
# One-Hot Encoding 등 전처리는 분할 후에 적용하는 것이 일반적입니다.
STRATA_VAR = amc_filtered['procedure_cd_fin']

# -------------------------------------------------------------
# 2. 층화 무작위 분할 수행 (70% Train, 30% Test)
# -------------------------------------------------------------

# stratify=STRATA_VAR 옵션을 사용하여 procedure_cd_fin의 비율을 Train/Test에 동일하게 유지합니다.
X_train_amc, X_test_amc, y_train_amc, y_test_amc = train_test_split(
    X_amc,
    y_amc,
    test_size=0.3, # 30%를 테스트 셋으로 사용
    random_state=RANDOM_STATE, # 무작위성 고정
    stratify=STRATA_VAR # procedure_cd_fin을 층화 기준으로 사용
)

# -------------------------------------------------------------
# 3. 결과 확인
# -------------------------------------------------------------
print("✅ Python의 train_test_split을 사용하여 층화 무작위 분할이 완료되었습니다.")
print(f"🔥 Train Data (특성) 크기: {X_train_amc.shape}")
print(f"🧪 Test Data (특성) 크기: {X_test_amc.shape}")

# (선택 사항) 층화 확인: procedure_cd_fin의 비율 비교
print("\n--- procedure_cd_fin 비율 비교 (Top 3) ---")
train_ratios = X_train_amc['procedure_cd_fin'].value_counts(normalize=True).head(3)
test_ratios = X_test_amc['procedure_cd_fin'].value_counts(normalize=True).head(3)
ratio_comparison = pd.DataFrame({'Train Ratio': train_ratios, 'Test Ratio': test_ratios})
print(ratio_comparison)

In [ ]:
# 2. 데이터 전처리 (제한된 컬럼에 맞춰 수정)
# ----------------------------------------------------------------------
def preprocess_data(df):
   
    # ⭐ 이진 (0/1) 플래그 컬럼 처리 ⭐
    binary_flag_cols = [
        'mi', 'pvd', 'mld', 'diabwc', 'rend', 'canc',
        'heparins', 'coumarin_deriatives'
    ]
   
    for col in binary_flag_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype(int)

    # ⭐ 범주형 변수 ⭐
    cat_features = ['surgeon_dept_cd', 'anesthesia_cd', 'procedure_cd_fin']
    for col in cat_features:
        if col in df.columns:
            df[col] = df[col].astype(str)

    target_procedures = ['5059'] # 평균 사용량 10이상인 수술 코드
    if 'procedure_cd_fin' in df.columns:
        for code in target_procedures:
            df[f'procedure_cd_fin_{code}'] = (df['procedure_cd_fin'] == code).astype(int)
        df = df.drop(columns = ['procedure_cd_fin'])
    df = pd.get_dummies(df, columns = ['surgeon_dept_cd', 'anesthesia_cd'])

    # ⭐ 수치형/임상 수치 컬럼 NaN 처리 ⭐
    numeric_features_for_imputation = ['age', 'Hb', 'PT_inr', 'Plt', 'aPTT']
   
    for col in numeric_features_for_imputation:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].mean())
       
    return df

X_train_amc = preprocess_data(X_train_amc)
X_test_amc = preprocess_data(X_test_amc)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


X_train_amc, X_test_amc = X_train_amc.align(X_test_amc, join='outer', axis = 1, fill_value = 0)

numeric_cols = ['age', 'Hb', 'PT_inr', 'Plt', 'aPTT']
cols_to_scale = [c for c in numeric_cols if c in X_train_amc.columns]

if cols_to_scale:
    scaler = StandardScaler()
    
    X_train_amc[cols_to_scale] = scaler.fit_transform(X_train_amc[cols_to_scale])
    X_test_amc[cols_to_scale] = scaler.transform(X_test_amc[cols_to_scale])



In [ ]:
X_train_amc = X_train_amc.astype(np.float32)
X_test_amc = X_test_amc.astype(np.float32)

In [ ]:
# diabwc 제거, 국소 마취 제거(5)
X_train_amc = X_train_amc[['Hb', 'PT_inr', 'Plt', 'aPTT', 'anesthesia_cd_1', 
       'anesthesia_cd_9', 'canc', 'heparins', 'msbos', 'pvd',
       'surgeon_dept_cd_ENT', 'surgeon_dept_cd_GS', 'surgeon_dept_cd_NS',
       'surgeon_dept_cd_OBGY', 'surgeon_dept_cd_OS', 'surgeon_dept_cd_PS',
       'surgeon_dept_cd_TS', 'surgeon_dept_cd_URO']]
X_test_amc = X_test_amc[['Hb', 'PT_inr', 'Plt', 'aPTT', 'anesthesia_cd_1',
       'anesthesia_cd_9', 'canc', 'heparins', 'msbos', 'pvd',
       'surgeon_dept_cd_ENT', 'surgeon_dept_cd_GS', 'surgeon_dept_cd_NS',
       'surgeon_dept_cd_OBGY', 'surgeon_dept_cd_OS', 'surgeon_dept_cd_PS',
       'surgeon_dept_cd_TS', 'surgeon_dept_cd_URO']]

n_features = len(list(X_train_amc))

In [ ]:
### AMC 모델


best_params_list = []
final_performance = pd.DataFrame()
predictions_df = pd.DataFrame({'Actual_Value': y_test_amc})

# ----------------------------------------------------------------------
# 5. 모델 학습, 예측 및 평가 실행 (훈련 성능 및 실행 시간 측정)
# ----------------------------------------------------------------------
print("--- 모델 학습 및 확장 튜닝 시작 ---\n")

for name, (model, param_grid) in models.items():
    print(f">>> {name} 학습 중...")
   
    start_time = time.time() # ⭐ 시간 측정 시작
   
    # 튜닝 및 학습
    if name == 'Linear Regression':
        trained_model = model.fit(X_train_amc, y_train_amc)
        best_params = 'Default'
    else:
        # ⭐ 튜닝 조합 수 계산 및 진행 정보 출력 ⭐
        param_counts = [len(v) for v in param_grid.values()]
        total_combinations = np.prod(param_counts)
       
        # --- CV 정보 고정 및 계산 (10-Fold, 10-Repeats 가정) ---
        n_splits_per_repeat = 10
        n_repeats = 10
        n_splits_total = n_splits_per_repeat * n_repeats # 100회
       
        # 총 Fit 횟수 계산 (총 조합 수 * 총 폴드 수)
        total_fits = total_combinations * n_splits_total
       
        print(f"\n      [CV 전략] {n_splits_per_repeat} K-Fold를 {n_repeats}회 반복 (총 {n_splits_total} 폴드)")
        print(f"      [Grid Search] 총 {total_combinations}개 조합 테스트 시작")
        print(f"      (전체 Fit 횟수: {total_fits}회)")
        
        grid_search = GridSearchCV(model, param_grid, cv=cv_strategy,
                                   scoring=scoring_metric, n_jobs=-1,verbose= 3)
        grid_search.fit(X_train_amc, y_train_amc)
        
        trained_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        
        best_params_list.append({
            'Model':name,
            'Best_Parameters': best_params
        })
        print(f"   Best Params: {grid_search.best_params_}")

    

    # ⭐ 훈련 데이터셋 성능 계산 및 출력 ⭐
    y_pred_train = trained_model.predict(X_train_amc)
    y_pred_train = np.maximum(y_pred_train, 0)
   
    train_rmse = np.sqrt(mean_squared_error(y_train_amc, y_pred_train))
    train_mae = mean_absolute_error(y_train_amc, y_pred_train)
    train_r2 = r2_score(y_train_amc, y_pred_train)
    train_adj_r2 = adjusted_r2(y_train_amc, y_pred_train, n_features)

    print(f"   [Train Set Performance]")
    print(f"     RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}, Adj_R2: {train_adj_r2:.4f}")
   
    # 테스트 데이터셋 부트스트랩 평가
    bootstrap_metrics_df = calculate_bootstrap_ci(trained_model, X_test_amc, y_test_amc,
                                                  n_bootstrap=100, confidence_level=95,
                                                  n_features=n_features)
   
    end_time = time.time() # ⭐ 시간 측정 종료
    execution_time = end_time - start_time
   
    # 최종 결과 DataFrame 업데이트
    bootstrap_metrics_df['Model'] = name
    bootstrap_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
   
    final_performance = pd.concat([final_performance, bootstrap_metrics_df], ignore_index=True)
   
    # 예측 결과 저장을 위한 예측 수행
    y_pred_full = trained_model.predict(X_test_amc)
    predictions_df[f'{name}_Prediction'] = y_pred_full
   
print("\n--- 모델 학습 및 부트스트랩 평가 완료 ---\n")


In [ ]:
pd.DataFrame(best_params_list).to_csv('best_params_amc.csv', index=0, encoding = 'utf-8-sig')
final_performance.to_csv('performance_amc.csv', index=0, encoding = 'utf-8-sig')

In [ ]:
# 통합 데이터
X_train_total = pd.concat([X_train_smc, X_train_amc]).reset_index(drop=True)
X_test_total = pd.concat([X_test_smc, X_test_amc]).reset_index(drop=True)
y_train_total = pd.concat([y_train_smc, y_train_amc]).reset_index(drop=True)
y_test_total = pd.concat([y_test_smc, y_test_amc]).reset_index(drop=True)

In [ ]:
n_features = len(list(X_train_total))

In [ ]:
# 통합 모델

best_params_list = []
final_performance = pd.DataFrame()
predictions_df = pd.DataFrame({'Actual_Value': y_test_total})

# ----------------------------------------------------------------------
# 5. 모델 학습, 예측 및 평가 실행 (훈련 성능 및 실행 시간 측정)
# ----------------------------------------------------------------------
print("--- 모델 학습 및 확장 튜닝 시작 ---\n")

for name, (model, param_grid) in models.items():
    print(f">>> {name} 학습 중...")
   
    start_time = time.time() # ⭐ 시간 측정 시작
   
    # 튜닝 및 학습
    if name == 'Linear Regression':
        trained_model = model.fit(X_train_total, y_train_total)
        best_params = 'Default'
    else:
        # ⭐ 튜닝 조합 수 계산 및 진행 정보 출력 ⭐
        param_counts = [len(v) for v in param_grid.values()]
        total_combinations = np.prod(param_counts)
       
        # --- CV 정보 고정 및 계산 (10-Fold, 10-Repeats 가정) ---
        n_splits_per_repeat = 10
        n_repeats = 10
        n_splits_total = n_splits_per_repeat * n_repeats # 100회
       
        # 총 Fit 횟수 계산 (총 조합 수 * 총 폴드 수)
        total_fits = total_combinations * n_splits_total
       
        print(f"\n      [CV 전략] {n_splits_per_repeat} K-Fold를 {n_repeats}회 반복 (총 {n_splits_total} 폴드)")
        print(f"      [Grid Search] 총 {total_combinations}개 조합 테스트 시작")
        print(f"      (전체 Fit 횟수: {total_fits}회)")
        
        grid_search = GridSearchCV(model, param_grid, cv=cv_strategy,
                                   scoring=scoring_metric, n_jobs=-1,verbose= 3)
        grid_search.fit(X_train_total, y_train_total)
        
        trained_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        
        best_params_list.append({
            'Model':name,
            'Best_Parameters': best_params
        })
        print(f"   Best Params: {grid_search.best_params_}")

    

    # ⭐ 훈련 데이터셋 성능 계산 및 출력 ⭐
    y_pred_train = trained_model.predict(X_train_total)
    y_pred_train = np.maximum(y_pred_train, 0)
   
    train_rmse = np.sqrt(mean_squared_error(y_train_total, y_pred_train))
    train_mae = mean_absolute_error(y_train_total, y_pred_train)
    train_r2 = r2_score(y_train_total, y_pred_train)
    train_adj_r2 = adjusted_r2(y_train_total, y_pred_train, n_features)

    print(f"   [Train Set Performance]")
    print(f"     RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}, Adj_R2: {train_adj_r2:.4f}")
   
    # 테스트 데이터셋 부트스트랩 평가
    bootstrap_metrics_df = calculate_bootstrap_ci(trained_model, X_test_total, y_test_total,
                                                  n_bootstrap=100, confidence_level=95,
                                                  n_features=n_features)
   
    end_time = time.time() # ⭐ 시간 측정 종료
    execution_time = end_time - start_time
   
    # 최종 결과 DataFrame 업데이트
    bootstrap_metrics_df['Model'] = name
    bootstrap_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
   
    final_performance = pd.concat([final_performance, bootstrap_metrics_df], ignore_index=True)
   
    # 예측 결과 저장을 위한 예측 수행
    y_pred_full = trained_model.predict(X_test_total)
    predictions_df[f'{name}_Prediction'] = y_pred_full
   
print("\n--- 모델 학습 및 부트스트랩 평가 완료 ---\n")


In [ ]:
pd.DataFrame(best_params_list).to_csv('best_params_total.csv', index=0, encoding = 'utf-8-sig')
final_performance.to_csv('performance_total.csv', index=0, encoding = 'utf-8-sig')

In [ ]:
# smc 모델 amc 성능

# ⭐ 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요! ⭐
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 500, 'subsample': 0.7},
    'SVR': {'C': 10, 'gamma': 'auto'},
    'ANN (MLP)': {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
final_train_performance = pd.DataFrame()

# ⭐ 전제 변수 (이전 단계에서 정의된 값을 사용해야 함) ⭐
# X_train, y_train, n_features 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("--- 최적 모델 기반 훈련 데이터셋 CI 계산 시작 ---")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n>>> {name} 재학습 및 훈련 CI 계산 중...")
   
    start_time = time.time()
   
    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()
    
    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025
            
        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(X_train_smc, y_train_smc)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    train_metrics_df = calculate_bootstrap_ci(
        trained_model,
        X_test_amc,
        y_test_amc,
        n_bootstrap=100,
        confidence_level=95,
        n_features=n_features
    )
   
    end_time = time.time()
    execution_time = end_time - start_time
   
    # 결과 저장
    train_metrics_df['Model'] = name
    train_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    final_train_performance = pd.concat([final_train_performance, train_metrics_df], ignore_index=True)
   
print("\n--- 훈련 데이터셋 CI 계산 완료 ---\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("### 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) ###")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
final_train_performance = final_train_performance[output_cols]
print(final_train_performance)#.to_markdown(index=False))

In [ ]:
## smc 모델 total test

# ⭐ 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요! ⭐
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 500, 'subsample': 0.7},
    'SVR': {'C': 10, 'gamma': 'auto'},
    'ANN (MLP)': {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
final_train_performance = pd.DataFrame()

# ⭐ 전제 변수 (이전 단계에서 정의된 값을 사용해야 함) ⭐
# X_train, y_train, n_features 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("--- 최적 모델 기반 훈련 데이터셋 CI 계산 시작 ---")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n>>> {name} 재학습 및 훈련 CI 계산 중...")
   
    start_time = time.time()
   
    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()
    
    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025
            
        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(X_train_smc, y_train_smc)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    train_metrics_df = calculate_bootstrap_ci(
        trained_model,
        X_test_total,
        y_test_total,
        n_bootstrap=100,
        confidence_level=95,
        n_features=n_features
    )
   
    end_time = time.time()
    execution_time = end_time - start_time
   
    # 결과 저장
    train_metrics_df['Model'] = name
    train_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    final_train_performance = pd.concat([final_train_performance, train_metrics_df], ignore_index=True)
   
print("\n--- 훈련 데이터셋 CI 계산 완료 ---\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("### 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) ###")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
final_train_performance = final_train_performance[output_cols]
print(final_train_performance)#.to_markdown(index=False))

In [ ]:
### amc 모델 smc 테스트

# ⭐ 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요! ⭐
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.7},
    'SVR': {'C': 10, 'gamma': 'scale'},
    'ANN (MLP)': {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
final_train_performance = pd.DataFrame()

# ⭐ 전제 변수 (이전 단계에서 정의된 값을 사용해야 함) ⭐
# X_train, y_train, n_features 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("--- 최적 모델 기반 훈련 데이터셋 CI 계산 시작 ---")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n>>> {name} 재학습 및 훈련 CI 계산 중...")
   
    start_time = time.time()
   
    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()
    
    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025
            
        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(X_train_amc, y_train_amc)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    train_metrics_df = calculate_bootstrap_ci(
        trained_model,
        X_test_smc,
        y_test_smc,
        n_bootstrap=100,
        confidence_level=95,
        n_features=n_features
    )
   
    end_time = time.time()
    execution_time = end_time - start_time
   
    # 결과 저장
    train_metrics_df['Model'] = name
    train_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    final_train_performance = pd.concat([final_train_performance, train_metrics_df], ignore_index=True)
   
print("\n--- 훈련 데이터셋 CI 계산 완료 ---\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("### 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) ###")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
final_train_performance = final_train_performance[output_cols]
print(final_train_performance)#.to_markdown(index=False))

In [ ]:
### amc 모델 total 테스트

# ⭐ 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요! ⭐
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.7},
    'SVR': {'C': 10, 'gamma': 'scale'},
    'ANN (MLP)': {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
final_train_performance = pd.DataFrame()

# ⭐ 전제 변수 (이전 단계에서 정의된 값을 사용해야 함) ⭐
# X_train, y_train, n_features 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("--- 최적 모델 기반 훈련 데이터셋 CI 계산 시작 ---")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n>>> {name} 재학습 및 훈련 CI 계산 중...")
   
    start_time = time.time()
   
    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()
    
    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025
            
        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(X_train_amc, y_train_amc)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    train_metrics_df = calculate_bootstrap_ci(
        trained_model,
        X_test_total,
        y_test_total,
        n_bootstrap=100,
        confidence_level=95,
        n_features=n_features
    )
   
    end_time = time.time()
    execution_time = end_time - start_time
   
    # 결과 저장
    train_metrics_df['Model'] = name
    train_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    final_train_performance = pd.concat([final_train_performance, train_metrics_df], ignore_index=True)
   
print("\n--- 훈련 데이터셋 CI 계산 완료 ---\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("### 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) ###")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
final_train_performance = final_train_performance[output_cols]
print(final_train_performance)#.to_markdown(index=False))

In [ ]:
## 통합 모델 smc 성능

# ⭐ 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요! ⭐
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 1000, 'subsample': 1.0},
    'SVR': {'C': 10, 'gamma': 'auto'},
    'ANN (MLP)': {'activation': 'tanh', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
final_train_performance = pd.DataFrame()

# ⭐ 전제 변수 (이전 단계에서 정의된 값을 사용해야 함) ⭐
# X_train, y_train, n_features 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("--- 최적 모델 기반 훈련 데이터셋 CI 계산 시작 ---")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n>>> {name} 재학습 및 훈련 CI 계산 중...")
   
    start_time = time.time()
   
    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()
    
    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025
            
        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(X_train_total, y_train_total)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    train_metrics_df = calculate_bootstrap_ci(
        trained_model,
        X_test_smc,
        y_test_smc,
        n_bootstrap=100,
        confidence_level=95,
        n_features=n_features
    )
   
    end_time = time.time()
    execution_time = end_time - start_time
   
    # 결과 저장
    train_metrics_df['Model'] = name
    train_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    final_train_performance = pd.concat([final_train_performance, train_metrics_df], ignore_index=True)
   
print("\n--- 훈련 데이터셋 CI 계산 완료 ---\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("### 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) ###")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
final_train_performance = final_train_performance[output_cols]
print(final_train_performance)#.to_markdown(index=False))

In [ ]:
## 통합 모델 amc 성능

# ⭐ 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요! ⭐
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 1000, 'subsample': 1.0},
    'SVR': {'C': 10, 'gamma': 'auto'},
    'ANN (MLP)': {'activation': 'tanh', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
final_train_performance = pd.DataFrame()

# ⭐ 전제 변수 (이전 단계에서 정의된 값을 사용해야 함) ⭐
# X_train, y_train, n_features 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("--- 최적 모델 기반 훈련 데이터셋 CI 계산 시작 ---")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n>>> {name} 재학습 및 훈련 CI 계산 중...")
   
    start_time = time.time()
   
    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()
    
    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025
            
        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(X_train_total, y_train_total)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    train_metrics_df = calculate_bootstrap_ci(
        trained_model,
        X_test_amc,
        y_test_amc,
        n_bootstrap=100,
        confidence_level=95,
        n_features=n_features
    )
   
    end_time = time.time()
    execution_time = end_time - start_time
   
    # 결과 저장
    train_metrics_df['Model'] = name
    train_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    final_train_performance = pd.concat([final_train_performance, train_metrics_df], ignore_index=True)
   
print("\n--- 훈련 데이터셋 CI 계산 완료 ---\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("### 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) ###")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
final_train_performance = final_train_performance[output_cols]
print(final_train_performance)#.to_markdown(index=False))